# 07 — Earth Engine: additional covariates and multi-year NPP

Extends the census table from `00_gee_extraction.ipynb` with layers that the
models and the crop light-use-efficiency estimate use. Same lattice, same
chunked and resumable `reduceRegions` approach, masked pixels excluded.

| Group | Layer | Use |
|---|---|---|
| `npp_years` | MOD17A3HGF NPP 2020–2024, each year (gC/m²/yr) | multi-year NPP target and inter-annual range |
| `s1` | Sentinel-1 GRD VV and VH backscatter (dB), kharif and rabi medians | model covariates (canopy structure, soil moisture) |
| `lst` | Landsat 8/9 Collection 2 land-surface temperature (°C), kharif, rabi and annual medians | model covariates (crop water status) |
| `par` | ERA5-Land monthly downward solar radiation → PAR (MJ/m²/month), Jun 2024 – May 2025 | crop light-use-efficiency NPP |

Writes `IIRS/Carbon_Stocks/v2/ludhiana_cells_extra.csv.gz`; `02_carbon_pipeline.ipynb`
uses it automatically when present.

In [1]:
import json
import sys
import time
from datetime import date
from pathlib import Path

import ee
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
import config as C

EE_PROJECT = "my-projects-510917"
NPP_YEARS = list(range(2020, 2025))
KHARIF = ("2024-06-01", "2024-11-01")      # rice season
RABI = ("2024-11-01", "2025-05-01")        # wheat season
CHUNK = 1500
SCALE = {"npp_years": 50, "s1": 20, "lst": 30, "par": 1000}

V2 = C.DATA / "v2"
PARTS = V2 / "chunks_extra"
PARTS.mkdir(parents=True, exist_ok=True)
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
grid = pd.read_csv(V2 / "v2_grid.csv") if (V2 / "v2_grid.csv").exists() else C.build_grid_v2()
bnd = C.load_boundary(C.BOUNDARY_DEFAULT).union(C.load_boundary(C.BOUNDARY_CHECK))
region = ee.Geometry.Rectangle(list(bnd.bounds))
print(f"{len(grid):,} cells")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


71,197 cells


In [2]:
# MOD17 NPP, every year 2020-2024 (scaled to gC/m2/yr; invalid DN masked)
def npp_year(y):
    im = ee.ImageCollection("MODIS/061/MOD17A3HGF").filterDate(f"{y}-01-01", f"{y + 1}-01-01").first().select("Npp")
    return im.updateMask(im.gte(-30000).And(im.lte(32700))).multiply(0.1).rename(f"npp_{y}")
npp_years = ee.Image.cat([npp_year(y) for y in NPP_YEARS])

# Sentinel-1 GRD, IW, VV+VH, both orbits; seasonal medians in dB
def s1_season(start, end, tag):
    col = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(region).filterDate(start, end)
           .filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    m = col.select(["VV", "VH"]).median()
    return m.rename([f"vv_{tag}", f"vh_{tag}"]).addBands(m.select("VH").subtract(m.select("VV")).rename(f"vhvv_{tag}"))
s1 = s1_season(*KHARIF, "kharif").addBands(s1_season(*RABI, "rabi"))

# Landsat 8/9 Collection 2 Level-2 surface temperature, cloud and shadow masked
def lst_col(start, end):
    def prep(im):
        qa = im.select("QA_PIXEL")
        clear = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0)).And(qa.bitwiseAnd(1 << 1).eq(0))
        return im.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).updateMask(clear)
    return (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
            .filterBounds(region).filterDate(start, end).map(prep))
lst = ee.Image.cat([lst_col(*KHARIF).median().rename("lst_kharif"), lst_col(*RABI).median().rename("lst_rabi"),
                    lst_col(KHARIF[0], RABI[1]).median().rename("lst_annual")])

# ERA5-Land monthly downward shortwave -> PAR (0.48 of shortwave), MJ/m2/month
months = pd.date_range(KHARIF[0], periods=12, freq="MS")
era = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
par = ee.Image.cat([era.filterDate(str(m.date()), str((m + pd.offsets.MonthBegin()).date())).first()
                    .select("surface_solar_radiation_downwards_sum").multiply(0.48 / 1e6)
                    .rename("par_" + m.strftime("%b%y")) for m in months])

IMAGES = {"npp_years": npp_years, "s1": s1, "lst": lst, "par": par}
for k, im in IMAGES.items():
    print(k, im.bandNames().getInfo())

npp_years ['npp_2020', 'npp_2021', 'npp_2022', 'npp_2023', 'npp_2024']


s1 ['vv_kharif', 'vh_kharif', 'vhvv_kharif', 'vv_rabi', 'vh_rabi', 'vhvv_rabi']


lst ['lst_kharif', 'lst_rabi', 'lst_annual']


par ['par_Jun24', 'par_Jul24', 'par_Aug24', 'par_Sep24', 'par_Oct24', 'par_Nov24', 'par_Dec24', 'par_Jan25', 'par_Feb25', 'par_Mar25', 'par_Apr25', 'par_May25']


In [3]:
def cell_fc(df):
    s = C.GRID_STEP_DEG
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Rectangle([c * s, r * s, (c + 1) * s, (r + 1) * s], None, False), {"cell_id": k})
        for k, r, c in zip(df.cell_id, df.grid_row, df.grid_col)])

def reduce_chunk(name, df, tries=4):
    img = IMAGES[name]
    fc = img.reduceRegions(collection=cell_fc(df), reducer=ee.Reducer.mean(), scale=SCALE[name], tileScale=4)
    for attempt in range(tries):
        try:
            out = ee.data.computeFeatures({"expression": fc, "fileFormat": "PANDAS_DATAFRAME"})
            return out.reindex(columns=["cell_id"] + img.bandNames().getInfo())
        except Exception as exc:
            wait = 15 * 2 ** attempt
            print(f"   {name}: {type(exc).__name__}: {str(exc)[:120]} - retry in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"{name} chunk failed {tries} times")

chunks = [grid.iloc[i:i + CHUNK] for i in range(0, len(grid), CHUNK)]
t0 = time.time()
tables = []
for name in IMAGES:
    for j, df in enumerate(chunks):
        part = PARTS / f"{name}_{j:03d}.csv"
        if not part.exists():
            reduce_chunk(name, df).to_csv(part, index=False)
    tab = pd.concat([pd.read_csv(p) for p in sorted(PARTS.glob(f"{name}_*.csv"))], ignore_index=True)
    assert tab.cell_id.is_unique and len(tab) == len(grid), f"{name}: {len(tab)} rows for {len(grid)} cells"
    tables.append(tab.set_index("cell_id"))
    print(f"{name:10s} done  (elapsed {(time.time() - t0) / 60:.1f} min)")
extra = pd.concat(tables, axis=1).reset_index()

npp_years  done  (elapsed 0.0 min)
s1         done  (elapsed 0.0 min)


lst        done  (elapsed 0.0 min)


par        done  (elapsed 0.0 min)


In [4]:
checks = {
    "NPP years within MOD17 scaled range": extra[[f"npp_{y}" for y in NPP_YEARS]].stack().dropna().between(-3000, 3270).all(),
    "S1 backscatter within -35..+5 dB": extra.filter(regex="^(vv|vh)_").stack().dropna().between(-35, 5).all(),
    "VH below VV": (extra["vhvv_kharif"].dropna() < 0).mean() > 0.95,
    "LST within 0..60 C": extra.filter(regex="^lst_").stack().dropna().between(0, 60).all(),
    "PAR within 50..400 MJ/m2/month": extra.filter(regex="^par_").stack().dropna().between(50, 400).all(),
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
print(extra.describe().T[["count", "mean", "min", "max"]].round(2).to_string())
assert all(checks.values()), "extra covariates failed validation"
extra.to_csv(V2 / "ludhiana_cells_extra.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})
man = json.loads((V2 / "manifest.json").read_text())
man["extra"] = {"run_date": str(date.today()), "npp_years": NPP_YEARS, "kharif": KHARIF, "rabi": RABI, "scales_m": SCALE,
                "s1": "COPERNICUS/S1_GRD IW VV+VH, seasonal median dB",
                "lst": "LANDSAT/LC08+LC09 C02 T1_L2 ST_B10 x 0.00341802 + 149 - 273.15, QA cloud/shadow/dilated-cloud masked, median",
                "par": "ECMWF/ERA5_LAND/MONTHLY_AGGR surface_solar_radiation_downwards_sum x 0.48 / 1e6"}
(V2 / "manifest.json").write_text(json.dumps(man, indent=2))
print("Wrote IIRS/Carbon_Stocks/v2/ludhiana_cells_extra.csv.gz", extra.shape)

PASS NPP years within MOD17 scaled range
PASS S1 backscatter within -35..+5 dB
PASS VH below VV
PASS LST within 0..60 C
PASS PAR within 50..400 MJ/m2/month


               count    mean     min     max
npp_2020     68404.0   85.18    0.00  217.70
npp_2021     68404.0  101.57    0.00  237.10
npp_2022     68404.0  127.05    0.00  262.60
npp_2023     68404.0  164.29    0.00  313.50
npp_2024     68404.0   95.85    0.00  240.20
vv_kharif    71197.0  -11.24  -21.98    0.44
vh_kharif    71197.0  -18.13  -29.63   -9.52
vhvv_kharif  71197.0   -6.89  -13.59   -4.11
vv_rabi      71197.0  -11.32  -21.43    0.19
vh_rabi      71197.0  -18.61  -29.84   -9.81
vhvv_rabi    71197.0   -7.29  -14.27   -4.02
lst_kharif   71197.0   33.35   28.16   47.81
lst_rabi     71197.0   25.18   19.32   38.17
lst_annual   71197.0   30.25   24.55   44.39
par_Jun24    71197.0  359.34  357.04  361.51
par_Jul24    71197.0  290.02  281.84  293.72
par_Aug24    71197.0  245.74  238.92  249.61
par_Sep24    71197.0  244.81  234.99  253.63
par_Oct24    71197.0  269.18  267.55  270.03
par_Nov24    71197.0  202.66  200.85  204.46
par_Dec24    71197.0  181.58  180.05  183.79
par_Jan25 

Wrote IIRS/Carbon_Stocks/v2/ludhiana_cells_extra.csv.gz (71197, 27)
